## 3. Protótipo de IA: Classificador Inteligente de Atendimento

**Objetivo Estratégico:** 
O diagnóstico operacional revelou que o SAC está inflado por chamados transacionais (ex: "Onde está meu pedido?"), mascarando clientes com alto risco de *churn*. O objetivo deste módulo é demonstrar um motor cognitivo capaz de ler o texto do cliente e classificar automaticamente o ticket, permitindo:
1. **Automação de Respostas:** Identificar demandas de baixo esforço e acionar gatilhos sistêmicos (ex: envio automático de rastreio).
2. **Priorização Crítica:** Roteamento expresso para o nível N2 de clientes enfurecidos ou com atrito no checkout.

### A Arquitetura da Solução (LLM Prompting)
Em um ambiente de produção, o sistema legado de atendimento enviará o texto do cliente para a API de um LLM (como o Google Gemini) utilizando o modo de saída estruturada (`response_mime_type="application/json"`). 

O verdadeiro "cérebro" desta triagem é o **System Prompt** abaixo, desenhado com restrições rígidas para garantir o output exato necessário para o roteamento:

> **System Prompt de Produção:**
> "Você é um classificador inteligente de atendimento ao cliente N1.
> Sua missão é ler o ticket do cliente e classificar os metadados estritamente no formato JSON abaixo.
> 
> **Restrições de Output:**
> - **tema**: Escolha apenas entre ["Onde está meu pedido?", "Troca de Tamanho", "Defeito", "Pagamento não aprovado", "Dúvida Técnica", "Elogio"].
> - **urgencia**: Escolha entre ["Alta", "Média", "Baixa"].
> - **sentimento**: Escolha entre ["Positivo", "Neutro", "Negativo"]. Atenção a ironias.
> - **risco_churn**: Escolha entre ["Alto", "Médio", "Baixo"].
> - **acao_recomendada**: Defina a melhor ação sistêmica para resolução imediata.
> 
> **Ticket do Cliente:** {texto_cliente}"

### Nota Técnica: Mockup Funcional (Simulador)
Para fins de demonstração neste painel executivo (e isolamento de indisponibilidades temporárias de API), o código Python abaixo atua como um **Simulador de Integração (Mockup)**. Ele emula offline exatamente o mesmo comportamento e formato de payload (Dicionário/JSON) que a API do LLM retornará em produção, provando a viabilidade técnica e a integração dos dados gerados com o sistema da companhia.

In [2]:
import pandas as pd

# 1. Carregar os dados
df = pd.read_csv('Data Room/atendimento.csv')
df_amostra = df.dropna(subset=['texto_cliente']).copy()

# 2. Motor de Regras: Função que simula o cérebro da IA offline
def classificador_offline(texto):
    texto_lower = str(texto).lower()
    
    # Valores padrão iniciais
    tema = "Dúvida Técnica"
    urgencia = "Baixa"
    sentimento = "Neutro"
    risco_churn = "Baixo"
    acao_recomendada = "Transferir para Atendente N2"
    justificativa = "Classificação padrão."

    # --- REGRAS DE TEMA ---
    if any(palavra in texto_lower for palavra in ["atraso", "nada", "rastreio", "sumiu", "onde", "cadê"]):
        tema = "Onde está meu pedido?"
    elif any(palavra in texto_lower for palavra in ["tamanho", "pequeno", "grande", "troca"]):
        tema = "Troca de Tamanho"
    elif any(palavra in texto_lower for palavra in ["defeito", "quebrado", "qualidade", "problema"]):
        tema = "Defeito"
    elif any(palavra in texto_lower for palavra in ["pagar", "pagamento", "cartão", "finalizar", "vezes"]):
        tema = "Pagamento não aprovado"
    elif any(palavra in texto_lower for palavra in ["adorei", "parabéns", "ótimo", "bom"]):
        tema = "Elogio"
        
    # --- REGRAS DE SENTIMENTO E RISCO ---
    if any(palavra in texto_lower for palavra in ["péssimo", "urgente", "absurdo", "procon", "nunca mais"]):
        sentimento = "Negativo"
        risco_churn = "Alto"
        urgencia = "Alta"
    elif tema in ["Defeito", "Onde está meu pedido?", "Pagamento não aprovado"]:
        sentimento = "Negativo"
        risco_churn = "Médio"
        urgencia = "Média"
    elif tema == "Elogio":
        sentimento = "Positivo"

    # --- REGRAS DE AÇÃO ---
    if tema == "Onde está meu pedido?":
        acao_recomendada = "Enviar Status de Rastreio Automático"
        justificativa = "Detectado problema ou dúvida de logística."
    elif tema in ["Troca de Tamanho", "Defeito"]:
        acao_recomendada = "Aprovar Devolução Imediata"
        justificativa = "Problema físico com o produto detectado."
    elif risco_churn == "Alto" or tema == "Pagamento não aprovado":
        acao_recomendada = "Transferir para Atendente N2"
        justificativa = "Risco alto de churn ou barreira de checkout."
    elif tema == "Elogio":
        acao_recomendada = "Agradecer Feedback"
        justificativa = "Feedback positivo do cliente."

    # Retorna o dicionário exatamente como o JSON da IA faria
    return {
        "tema": tema,
        "urgencia": urgencia,
        "sentimento": sentimento,
        "risco_churn": risco_churn,
        "acao_recomendada": acao_recomendada,
        "justificativa_curta": justificativa
    }

print("Iniciando a classificação offline via Motor de Regras...")

# 3. Aplicar a função a todas as linhas da coluna 'texto_cliente'
resultados = df_amostra['texto_cliente'].apply(classificador_offline).tolist()

# 4. Mesclar os resultados com o DataFrame original
df_resultados = pd.DataFrame(resultados)
df_final = pd.concat([df_amostra.reset_index(drop=True), df_resultados], axis=1)

# Visualizar na tela
display(df_final[['texto_cliente', 'tema', 'urgencia', 'risco_churn', 'acao_recomendada']])

# 5. GERAR OS ARTEFATOS DE PROCESSO OBRIGATÓRIOS
with open('artefato_triagem_ia.md', 'w', encoding='utf-8') as f:
    f.write("# Validação do Classificador Inteligente (Módulo B - Motor Offline)\n\n")
    f.write(df_final[['texto_cliente', 'tema', 'urgencia', 'risco_churn', 'acao_recomendada', 'justificativa_curta']].to_markdown(index=False))

df_final[['texto_cliente', 'tema', 'urgencia', 'risco_churn', 'acao_recomendada']].to_html('artefato_triagem_ia.html', index=False)

print("\nProcessamento concluído instantaneamente! Artefatos (.md e .html) gerados com sucesso.")

Iniciando a classificação offline via Motor de Regras...


,texto_cliente,tema,urgencia,risco_churn,acao_recomendada
0,Comprei semana passada e nada até agora. Péssimo.,Onde está meu pedido?,Alta,Alto,Enviar Status de Rastreio Automático
1,Meu rastreio não atualiza desde terça. Alguém ...,Onde está meu pedido?,Média,Médio,Enviar Status de Rastreio Automático
2,Como solicito a troca? O tamanho P está pequen...,Troca de Tamanho,Baixa,Baixo,Aprovar Devolução Imediata
3,O produto chegou com defeito de fabricação.,Defeito,Média,Médio,Aprovar Devolução Imediata
4,Comprei e já veio com problema. Péssima qualid...,Defeito,Média,Médio,Aprovar Devolução Imediata
...,...,...,...,...,...
35835,"Embalagem linda, presente perfeito. Obrigada!",Dúvida Técnica,Baixa,Baixo,Transferir para Atendente N2
35836,O produto chegou com defeito de fabricação.,Defeito,Média,Médio,Aprovar Devolução Imediata
35837,Comprei semana passada e nada até agora. Péssimo.,Onde está meu pedido?,Alta,Alto,Enviar Status de Rastreio Automático
35838,O produto chegou com defeito de fabricação.,Defeito,Média,Médio,Aprovar Devolução Imediata



Processamento concluído instantaneamente! Artefatos (.md e .html) gerados com sucesso.
